In [ ]:
"""
sandbox.ipynb

A sandbox to play around with new analyses.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-03-23
Python Version: 3.11.14
"""

from squiggs.neuron_viewer import NeuronViewer
from squiggs.renderers import FitRenderer
from sg.fitter import LVMFamily
from sg.eval_models import plot_latents

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
"""
TODO:
backburner
- what about adding ReLU to the taskvar model and removing block?
- plot rasters, sanity check the psths
"""

In [ ]:
# check if you can get back m2 z-score psths from dividing tuber
# ask joao about the masking of the problem via zscore

In [ ]:
"""
1. try grid search both with and without refitting 
2. combine dms and dls
"""

## Fit 

In [ ]:
subj_id = "MM012"
sess_id = "20231219_130847"

family = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=2,
    n_latents_addt=2,
    sanity_check=0,
    n_splines=5,
    task_vars={
        "digital": [
            "response",
            "rewarded",
            "block_side",
            "strategy",
            "response_prev",
            "rewarded_prev",
        ],
        "analog": [],
    },
    tpre=0.5,
    tpost=1,
)
family.fit_all()
family.eval()

In [ ]:
family_refit = LVMFamily(
    trial_data=family.trial_data,
    spike_times=family.spike_times,
    session_data=family.session_data,
    regions=family.regions,
    n_latents_mult=1,
    n_latents_addt=1,
    sanity_check=0,
    task_vars={
        "digital": [
            "response",
            "rewarded",
            "block_side",
            "strategy",
            "response_prev",
            "rewarded_prev",
        ],
        "analog": [],
    },
    tpre=0.5,
    tpost=1,
    refit=True,
    max_iter=10,
    norm_activity=True,
)
# family.get_data()
family.fit_all()
family.eval()
family_refit.fit_all()
family_refit.eval()

In [ ]:
from sg.fitter import Encoder

encoder_pr = Encoder(
    trial_data=family.trial_data,
    spike_times=family.spike_times,
    session_data=family.session_data,
    regions=family.regions,
    task_vars={
        "digital": [
            "response",
            "rewarded",
            "block_side",
            "strategy",
            "response_prev",
            "rewarded_prev",
        ],
        "analog": [
            "pr",
        ],
    },
    tpre=0.5,
    tpost=1,
    norm_activity=True,
)

encoder_pr.get_data()
encoder_pr.fit_baseline()
encoder_pr.fit_taskvar()
encoder_pr.eval()

In [ ]:
trial_data_latent = family.trial_data
trial_data_latent["latent_g0"] = family.mod_gain.gain_mu.get_weights()
trial_data_latent["latent_a0"] = family.mod_offset.offset_mu.get_weights()

In [ ]:
encoder_latent = Encoder(
    trial_data=family.trial_data_latent,
    spike_times=family.spike_times,
    session_data=family.session_data,
    regions=family.regions,
    task_vars={
        "digital": [
            "response",
            "rewarded",
            "block_side",
            "strategy",
            "response_prev",
            "rewarded_prev",
        ],
        "analog": [
            "latent_a0",
        ],
    },
    tpre=0.5,
    tpost=1,
    norm_activity=True,
)

encoder_latent.get_data()
encoder_latent.fit_baseline()
encoder_latent.fit_taskvar()
encoder_latent.eval()

In [ ]:
mode = "offset"
model = family_refit.mod_gain if mode == "gain" else family.mod_offset

coupling = (
    model.readout_gain.weight.data[:].T
    if mode == "gain"
    else model.readout_offset.weight.data
)

beta_pr = encoder_pr.mod_taskvar.tv.weight.data[:][-1]
beta_latent = encoder_latent.mod_taskvar.tv.weight.data[:][-1]

In [ ]:
# actually a VAE???

In [ ]:
plt.figure()
plt.scatter(
    coupling,
    beta_latent[family.cids],
    s=0.5,
    cmap="coolwarm",
    vmin=-0.8,
    vmax=0.8,
    c=(family.res_taskvar["r2test"]),
)
plt.xlabel("coupling weight of lvm")
plt.ylabel("coupling weight of latent in encoder")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
from sg.fitlvm_utils import rsquared

latent_a0 = trial_data_latent["latent_a0"]

mode = "offset"
model = family.mod_gain if mode == "gain" else family.mod_offset

coupling = (
    model.readout_gain.weight.data[:].T
    if mode == "gain"
    else model.readout_offset.weight.data
)

sample = family.val_dl.dataset[:]
robs_ = sample["robs"].detach().cpu()
rhat_ = family.mod_taskvar(sample).detach().cpu()
h_ = (coupling.T.detach().cpu().numpy() * np.array(latent_a0)).T

rhat_offset_ = rhat_ + h_

r2test_posthum_latent = rsquared(
    robs_[:, model.cids], rhat_offset_, sample["dfs"][:, model.cids].detach().cpu()
)

In [ ]:
rhat_offset = family.mod_offset(sample).detach().cpu()

In [ ]:
reparam_param = family.mod_offset.offset_mu.weight[sample["indices"], :]
zh = family.mod_offset.reparameterize(reparam_param, family.mod_offset.logvar_h)
# the above two variables are identical during testing
h = family.mod_offset.readout_offset(zh)
# h is identical to h_

# the outputs are weirdly different, not a constant offset but it differs across the trial
# i bet it's the drift actually <- yup, definitely
# freeze the drift?
# sel.bias is a teeny bit different
# i think that self.tv.bias is also a bit different

In [ ]:
drift_tv = family.mod_taskvar.drift(sample["tents"]).detach().cpu().numpy()
drift_offset = family.mod_offset.drift(sample["tents"]).detach().cpu().numpy()
plt.figure()
plt.imshow(drift_tv - drift_offset)

In [ ]:
# is x before offset_mu identical? no, and it is offset by a bias
x = family.mod_offset.tv(sample["tv"])

plt.figure()
plt.imshow(
    (rhat_offset_ - family.mod_offset(sample).detach().cpu()).T, interpolation=None
)

In [ ]:
plt.figure()
plt.imshow(h.detach().cpu().numpy() - h_, cmap="magma")

In [ ]:
plt.figure()
plt.plot(reparam_param.detach().cpu().numpy())
plt.plot(zh.detach().cpu().numpy())
plt.plot(h.detach().cpu().numpy()[:, 0])

In [ ]:
plt.figure()
plt.scatter(family.res_offset["r2test"], r2test_posthum_latent, s=0.5, color="#C09815")
plt.plot([-0.1, 1], [-0.1, 1], color="#555555")

In [ ]:
tv_weights_tv = family.mod_taskvar.tv.weight.data[:].T
tv_weights_offset = family.mod_offset.tv.weight.data[:].T

plt.figure()
plt.imshow(tv_weights_offset - tv_weights_tv, aspect="auto")
plt.colorbar()

In [ ]:
(coupling.T.detach().cpu().numpy() * np.array(latent_a0)).shape

In [ ]:
rhat_.shape

In [ ]:
A = np.arange(10).reshape(-1, 1)
B = np.arange(3) * -1

A.shape, B.shape

In [ ]:
rhat_.shape

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR

reg = "ACC"
mode = "response"

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family.trial_data, mode=mode),
    spike_times=family.spike_times[reg],
    peths=get_psths_cond(family.psths[reg], family.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / subj_id / sess_id / reg / mode,
)

nv = NeuronViewer(
    num_units=family.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)

In [ ]:
np.where(
    [
        np.where(family.psths["ACC"][i] == 0)[0].shape[0] / (323 * 59) > 0.95
        for i in range(len(family.psths["ACC"]))
    ]
)

In [ ]:
plt.figure()
plt.imshow(family.psths["ACC"][0])

In [ ]:
plt.figure()
plt.plot(family.robs[:, 1])

In [ ]:
renderer = FitRenderer(
    family.mod_taskvar,
    x=family.test_dl.dataset[:],
    y=family.robs,
    save_subdir=Path("model_fits") / "0312-lm" / "offset",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)

In [ ]:
from sg.fitlvm_utils import eval_model

res_taskvar = eval_model(family.mod_taskvar, family.data_gd, family.test_dl.dataset)

In [ ]:
res_taskvar["r2test"][:10]

In [ ]:
family.get_cids()
family.cids_tv, family.cids_pca

In [ ]:
renderer = FitRenderer(
    family.mod_taskvar,
    x=family.test_dl.dataset[:],
    y=family.robs,
    save_subdir=Path("model_fits") / "0312-lm" / "offset",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)

In [ ]:
from core.data import get_pr

pr = get_pr(family.psths, family.regions, family.num_units)

In [ ]:
_ = plot_latents(
    family_refit,
    model=family_refit.mod_offset,
    potato=pr,
    label="popl resp",
    mode="offset",
)
_ = plot_latents(
    family_refit, model=family_refit.mod_gain, potato=pr, label="popl resp", mode="gain"
)

In [ ]:
_ = plot_latents(
    family, model=family.mod_offset, potato=(pr), label="popl resp", mode="offset"
)
_ = plot_latents(
    family, model=family.mod_gain, potato=(pr), label="popl resp", mode="gain"
)

In [ ]:
from sg.eval_models import plot_r2_comp

plot_r2_comp(
    family.res_taskvar,
    family.res_gain,
    label_a="",
    label_b="",
    title="",
    mode="unity",
    save=False,
    fpath=None,
)

### Delta R2, R2

In [ ]:
from sg.eval_models import monkey_cv_d_r2, plot_cv_d_r2

scramble_r2s, scramble_r2s_summary = monkey_cv_d_r2(
    family.trial_data, family.spike_times, family.session_data, family.regions
)
plot_cv_d_r2(scramble_r2s_summary)

### Strategy Decoding from Latents

In [ ]:
from sg.eval_models import latent_decoding, get_scores_mean
from pprint import pprint

scores = latent_decoding(family)
pprint(get_scores_mean(scores))

### cweights

In [ ]:
task_vars = family.task_vars
task_vars_str = {
    "response": ["left", "right"],
    "rewarded": ["no", "yes"],
    "block_side": ["left", "right"],
    "strategy": ["MF", "MB"],
    "response_prev": ["left", "none", "right"],
    "rewarded_prev": ["no", "yes"],
}

In [ ]:
import matplotlib.colors as colors
import matplotlib.cm as cm


def plot_cweights(family, mode, ax0=0, ax1=1):

    model = family.mod_gain if mode == "gain" else family.mod_offset
    coupling = (
        model.readout_gain.weight.data[:].T
        if mode == "gain"
        else mode.readout_offset.weight.data
    )

    i = 0
    for pivot in task_vars_str.keys():
        fig, axes = plt.subplots(
            ncols=len(task_vars_str[pivot]), figsize=(2 * len(task_vars_str[pivot]), 2)
        )

        for val, ax in zip(task_vars_str[pivot], axes.flat):
            taskvar_weights = family.mod_taskvar.tv.weight.data[:][i]
            norm = colors.Normalize(
                vmin=taskvar_weights.min(), vmax=taskvar_weights.max()
            )
            cmap = cm.viridis
            mapped_colors = cmap(norm(taskvar_weights))

            ax.scatter(coupling[:, ax0], coupling[:, ax1], color=mapped_colors, s=0.5)

            ax.set_xlabel("Latent 1")
            ax.set_ylabel("Latent 2")
            ax.set_title(f"{pivot}_{val}")

            i += 1

        fig.suptitle(pivot)
        fig.tight_layout()


plot_cweights(family, mode="gain", ax0=0, ax1=1)

In [ ]:
from sg.eval_models import plot_cweights_mult

plot_cweights_mult(family)

### gs n. latents 

In [ ]:
import importlib
import sg

importlib.reload(sg.eval_models)

In [ ]:
from sg.eval_models import get_latent_r

# max_n_latents = 0


def get_r2s_helper(subj_id, sess_id, regions):
    r2s = {}
    for reg in regions:
        r2s[reg] = get_latent_r(
            subj_id,
            sess_id,
            n_m=0,
            n_a=5,
            region=reg,
            do_plot=True,
            do_save=True,
        )
    return r2s


subj_id = "MR82"
sess_id = "20251027_152036"
r2s = get_r2s_helper(
    subj_id,
    sess_id,
    regions=["all", "DLS", "DMS"],
)

In [ ]:
import pickle
from utils.paths import PROJECT_ROOT

region = "DLS"

with open(
    PROJECT_ROOT.parents[0]
    / f"vars/families/{subj_id}/{sess_id}/no_pr/{region}/family-m0a1.pkl",
    "rb",
) as f:
    family_mac = pickle.load(f)

In [ ]:
family.psths.keys()

In [ ]:
family.trial_data

In [ ]:
get_choice_ts(family.trial_data, mode="response")

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR
from pathlib import Path

reg = "DLS"
mode = "response"

family = family_mac

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family.trial_data, mode=mode),
    spike_times=family.spike_times[reg],
    peths=get_psths_cond(family.psths[reg], family.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / family.subj_id / family.sess_id / reg / mode,
)

nv = NeuronViewer(
    num_units=family.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)

In [ ]:
from core.data import load_sess

subj_ids = ["MR82", "MR83", "MR85"]
sess_idxs = [5, 5, 5]
modes = ["new", "new", "new"]

folders = ["all", "ACC", "DMS", "M2", "DLS"]


def get_r2s_helper(subj_id, sess_idx, mode, folders, n_m=5):
    r2s = {}
    family.spike_times, family.trial_data, family.session_data, family.regions = (
        load_sess(subj_id=subj_id, sess_idx=sess_idx, mode=mode)
    )
    for f in folders:
        r2s[f] = get_latent_r(
            family.trial_data,
            family.spike_times,
            family.session_data,
            family.regions,
            subj_id=subj_id,
            sess_idx=sess_idx,
            folder=f,
            n_m=n_m,
            do_plot=True,
        )

In [ ]:
from pprint import pprint

for i in range(3):
    unit_spike_times, trial_data, session_data, regions = load_sess(
        subj_id=subj_ids[i], sess_idx=sess_idxs[i], mode=modes[i]
    )
    family = LVMFamily(
        trial_data=family.trial_data,
        spike_times=family.spike_times,
        session_data=family.session_data,
        regions=family.regions,
        n_latents_mult=1,
        n_latents_addt=1,
        task_vars={
            "digital": [
                "response",
                "rewarded",
                "block_side",
                "strategy",
                "response_prev",
                "rewarded_prev",
            ],
            "analog": [],
        },
        refit=True,
        max_iter=10,
        norm_activity=True,
    )
    family.get_data()
    pprint({reg: len(family.psths[reg]) for reg in regions})

In [ ]:
for i in range(3):
    get_r2s_helper(
        subj_ids[i], sess_idxs[i], modes[i], folders=["all", "DLS", "DMS"], n_m=5
    )

### res dfs filtering stuff

In [ ]:
from sg.eval_models import get_res

res = get_res(family, mode="taskvar")
radish = np.mean(res, axis=1)
bacon = np.std(res, axis=1)

In [ ]:
# 1. why is res.shape = (332, 150) but dfs.shape = (332, 154)
# -> how is robs filtered?
# 2. is the filtered array (with the offending neurons filtered out) passed to the lvms?
#    -> because it's still learning a major affect present in only 6 neurons

In [ ]:
res.shape

In [ ]:
np.where(np.abs(res[149, :]) > 5)

In [ ]:
plt.figure()
plt.hist(res[149, :])
plt.show()

In [ ]:
family.sample["dfs"][
    :, family.cids
].shape  # = robs.shape, makes sense beceause robs is filtered with cids

In [ ]:
plt.figure()
plt.plot(radish)
plt.fill_between(np.arange(len(radish)), radish - bacon, radish + bacon, alpha=0.5)
plt.show()

In [ ]:
plt.figure()
plt.imshow(family.data_gd.covariates["dfs"])
plt.show()

In [ ]:
_ = plot_latents(family, model=family.mod_offset, potato=bacon, mode="offset")

In [ ]:
plt.figure()
plt.imshow(family.robs.T, aspect="auto")
plt.show()